RECRIANDO O CORPUS E O PRÉ-PROCESSAMENTO:

In [ ]:
docs <- c(
    d1 = "recuperacao de informacao ordena documentos por relevancia",
    d2= "o modelo de espaco vetorial representa documentos como vetores",
    d3 = "bm25 e um modelo probabilistico de ranqueamento de texto",
    d4 = "aprendizado estatistico fundamenta a recuperacao moderna",
    d5 = "o indice invertido acelera a busca em muitos documentos",
    d6 = "embeddings capturam a semantica de palavras e documentos",
    d7 = "a avaliacao mede a relevancia dos resultados da busca",
    d8 = "ciencia de dados combina estatistica e programacao"
)

tok <- function (x) unlist(strsplit(tolower(x), "\\s+"))
tokens <- lapply(docs, tok)
vocab <- sort(unique(unlist(tokens)))

MONTANDO A MATRIZ DE FREQUÊNCIA, O TAMANHO DOS DOCUMENTOS E O IDF PROBABILÍSTICO:

In [ ]:
tf <- sapply(tokens, function(t) as.integer(table(factor(t, levels = vocab))))
rownames(tf) <- vocab
d1 <- colSums(tf)
avgdl <- mean(d1)

N <- ncol(tf)
df <- rowSums(tf > 0)
idf <- log(N - df + 0.5) / (df +0.5 + 1)

IMPLEMENTANDO A FUNÇÃO BM_25 DOC:

In [ ]:
bm25_doc <- function(termos, d, k1, b) {
  s <- 0
  for (t in termos) {
    if (!(t %in% vocab)) next
    f <- tf [t, d]
    if (f == 0) next
    K <- k1 * (1 - b + b * d1[d] / avgdl)
    s <- s + unname(idf[t]) * (f * (k1 + 1)) / (f + K)
  }
  unname(s)
}



TAREFA 1: IMPLEMENTANDO O BM25 E RANQUEANDO AS 3 CONSULTAS:

In [ ]:
consultas <- c("modelo de recuperacao", "busca e relevancia dos documentos", "estatistica e ciencia de dados")

for (cons in consultas) {
  termos <- tok(cons)
  scores <- sapply(colnames(tf), function(d) bm25_doc(termos, d, 1.2, 0.75))
  cat("\nConsulta:", cons, "\n")
  print(round(sort(scores, decreasing = TRUE), 3))
}


Consulta: modelo de recuperacao 
   d1    d3    d2    d4    d8    d6    d5    d7 
0.767 0.765 0.692 0.596 0.203 0.193 0.000 0.000 

Consulta: busca e relevancia dos documentos 
   d7    d1    d5    d6    d8    d3    d2    d4 
1.784 0.852 0.769 0.652 0.399 0.360 0.260 0.000 

Consulta: estatistica e ciencia de dados 
   d8    d3    d6    d1    d2    d4    d5    d7 
3.151 0.616 0.572 0.203 0.183 0.000 0.000 0.000 


COMPARANDO COM O TF-IDF DA AULA 02:

In [ ]:
idf_classico <- log(N / df)
w <- tf * idf_classico
cosseno <- function(a, b) sum(a * b) / (sqrt(sum(a^2)) * sqrt(sum(b^2)))

for(cons in consultas) {
  termos <- tok(cons)
  q <- as.integer(table(factor(termos, levels = vocab)))
  qw <- q * idf_classico
  scores <- apply(w, 2, function(dvec) cosseno(qw, dvec))
  cat("\nConsulta:", cons, "\n")
  print(round(sort(scores, decreasing = TRUE), 3))
}


Consulta: modelo de recuperacao 
   d1    d3    d4    d2    d6    d8    d5    d7 
0.254 0.233 0.215 0.208 0.025 0.023 0.000 0.000 

Consulta: busca e relevancia dos documentos 
   d7    d1    d5    d6    d8    d3    d2    d4 
0.503 0.185 0.151 0.106 0.065 0.062 0.030 0.000 

Consulta: estatistica e ciencia de dados 
   d8    d3    d6    d1    d2    d4    d5    d7 
0.788 0.074 0.071 0.014 0.011 0.000 0.000 0.000 


TAREFA 3; VARIANDO K1 E B:

In [ ]:
combinacoes <- list(
  list(k1 = 0,    b = 0.75),
  list(k1 = 1.2,  b = 0),
  list(k1 = 1.2,  b = 0.75),
  list(k1 = 1.2,  b = 1),
  list(k1 = 3,    b = 0.75)
)

consulta_teste <- tok("modelo de recuperacao")
for (comb in combinacoes) {
  scores <- sapply(colnames(tf), function(d) bm25_doc(consulta_teste, d, comb$k1, comb$b))
  cat(sprintf("\nk1=%.1f, b=%.2f\n", comb$k1, comb$b))
  print(round(sort(scores, decreasing = TRUE), 3))
}


k1=0.0, b=0.75
   d1    d2    d3    d4    d6    d8    d5    d7 
0.728 0.728 0.728 0.535 0.193 0.193 0.000 0.000 

k1=1.2, b=0.00
   d3    d1    d2    d4    d6    d8    d5    d7 
0.800 0.728 0.728 0.535 0.193 0.193 0.000 0.000 

k1=1.2, b=0.75
   d1    d3    d2    d4    d8    d6    d5    d7 
0.767 0.765 0.692 0.596 0.203 0.193 0.000 0.000 

k1=1.2, b=1.00
   d1    d3    d2    d4    d8    d6    d5    d7 
0.781 0.754 0.681 0.619 0.207 0.193 0.000 0.000 

k1=3.0, b=0.75
   d3    d1    d2    d4    d8    d6    d5    d7 
0.792 0.783 0.680 0.622 0.207 0.193 0.000 0.000 
